# 3.2. GPT-2 inference

In [ ]:
# The format is: [TOX]text[/TOX]»»[DETOX]text[/DETOX]
# [TOX]   - source text
# [DETOX] - target text
# »»      - separator

# So, we will add 5 custom tokens to the vocabulary:
# [TOX], [/TOX], [DETOX], [/DETOX], »»
tokens_dict = {
    "tox_begin": "[TOX]",
    "tox_end": "[/TOX]",
    "detox_begin": "[DETOX]",
    "detox_end": "[/DETOX]",
    "separator": "»»",
    "split": "[SPLIT]",
}

In [ ]:
KAGGLE_INFER = False

In [ ]:
if KAGGLE_INFER:
    model_output_dir = "/kaggle/working/models/gpt2-based"
else:
    model_output_dir = "../models/gpt2-based"

In [ ]:
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
)

In [ ]:
# Load the model and tokenizer
tokenizer = GPT2Tokenizer.from_pretrained(model_output_dir)
model = GPT2LMHeadModel.from_pretrained(model_output_dir)

In [ ]:
# Inference pipeline
from transformers import pipeline

generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device="cuda")

In [ ]:
def generate(text: str) -> list:
    """
    Generates suggestions for the given text.
    """
    prompt = (
        tokens_dict["tox_begin"]
        + text
        + tokens_dict["tox_end"]
        + tokens_dict["separator"]
        + tokens_dict["detox_begin"]
    )

    output = generator(prompt, max_length=len(prompt) * 2.5)[0]["generated_text"]

    suggestions = output.split(tokens_dict["separator"] + tokens_dict["detox_begin"])[1]
    
    # Replace all the special tokens with [SPLIT]
    for token in tokens_dict.values():
        suggestions = suggestions.replace(token, tokens_dict["split"])

    # Split by [SPLIT]
    suggestions = suggestions.split(tokens_dict["split"])

    # Trim if any of \", \n, or whitespace is present
    suggestions = [s.strip('"\n ') for s in suggestions]

    # Remove empty strings
    suggestions = list(filter(None, suggestions))

    return suggestions

In [ ]:
prompt = "Go and die, stupid!"

for suggestion in generate(prompt):
    print(suggestion)